In [ ]:
# ----------------------------------------------------------------------
# K-Fold 교차 검증 (Cross-Validation) 정의 및 원리
# ----------------------------------------------------------------------

# 1. 정의:
# - 전체 학습 데이터셋을 'K'개의 동일 크기 부분 집합(Fold)으로 분할하여 K번의 학습/평가를 반복 수행하는 검증 방법.
# - 최종 성능은 K번의 평가 점수 평균으로 계산됨.

# 2. 목적:
# - 일반화 성능 평가: 모델이 특정 데이터에 과적합되지 않고, 미지의 데이터에 얼마나 잘 작동하는지 객관적으로 측정.
# - 데이터 효율성: 모든 데이터 포인트가 한 번씩 검증 세트로, K-1번씩 학습 세트로 사용되어 데이터 활용률 극대화.
# - 편향 최소화: 단일 Train/Test 분할의 문제점(운에 따른 편향된 분할)을 해소.

# 3. 작동 원리 (K=5 예시):
# - 데이터 분할: 전체 데이터를 Fold 1, Fold 2, Fold 3, Fold 4, Fold 5로 나눔.
# - 반복 수행:
#   - 1회차: Fold 1을 검증(Test), 나머지 4개 Fold를 학습(Train)에 사용.
#   - 2회차: Fold 2를 검증(Test), 나머지 4개 Fold를 학습(Train)에 사용.
#   - ... 총 K번 반복.
# - 최종 결과: K번의 평가 점수(예: Accuracy)를 평균하여 모델의 최종 성능으로 산출.

# 4. 주요 고려 사항:
# - K 값 선택: 일반적으로 K=5 또는 K=10이 표준적으로 사용됨.
#   - K가 클수록: 학습 데이터량이 많아져 편향(Bias)은 낮아지나, 연산 시간이 길어짐.
# - Stratified K-Fold: 분류(Classification) 문제 시 필수. 타겟 변수(종속 변수)의 클래스 비율이 각 Fold에 균등하게 포함되도록 분할하여 편향을 방지.

In [2]:
# --------------------
# 라이브러리 및 데이터 준비
# --------------------
import numpy as np
from sklearn.model_selection import KFold, cross_val_score # KFold와 성능 평가 함수
from sklearn.linear_model import LogisticRegression # 예시 모델 (로지스틱 회귀)
from sklearn.datasets import load_iris

# Iris 데이터셋 로드
iris = load_iris()
X, y = iris.data, iris.target

print("--- 데이터 준비 완료 ---")
# X, y # 확인만

--- 데이터 준비 완료 ---


In [3]:
# --------------------
# 2단계: KFold 객체 정의 및 교차 검증 실행
# --------------------
# KFold(n_splits=5): 데이터를 5개의 Fold로 나눔 (K=5)
# shuffle=True: 데이터를 먼저 무작위로 섞어 데이터 편향 방지
kf = KFold(n_splits=5, shuffle=True, random_state=42) 

# 사용할 모델 정의
model = LogisticRegression(solver='liblinear', random_state=42)

# cross_val_score 실행: 
# - 모델(estimator)과 데이터(X, y)를 받음
# - KFold 객체(cv=kf)에 따라 5번 학습 및 평가 진행
# - 평가 지표(scoring='accuracy')를 사용
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print("\n--- 2단계: 5-Fold 교차 검증 점수 ---")
# 5번의 반복에서 얻은 각각의 정확도 점수
print(cv_scores)


--- 2단계: 5-Fold 교차 검증 점수 ---
[1.         0.93333333 0.93333333 0.96666667 0.96666667]


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial lo

In [4]:
# --------------------
# 3단계: 최종 성능 확인 (평균 점수)
# --------------------
mean_score = np.mean(cv_scores)
std_score = np.std(cv_scores)

print(f"\n--- 3단계: 최종 일반화 성능 ---")
print(f"평균 정확도 (Mean Accuracy): {mean_score:.4f}")
print(f"점수의 표준 편차 (Std Dev): {std_score:.4f}")

# 해석: 표준 편차가 낮을수록 모델의 성능이 데이터 분할에 따라 크게 변동하지 않고 안정적임을 의미합니다.


--- 3단계: 최종 일반화 성능 ---
평균 정확도 (Mean Accuracy): 0.9600
점수의 표준 편차 (Std Dev): 0.0249


In [ ]:
# --------------------------------------------------------
# health_survey

In [5]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression # cross_val_score를 위해 사용

# --------------------
# 데이터 로드 및 전처리
# --------------------

# 데이터 로드
df = pd.read_csv("health_survey.csv")

# 범주형 변수에 대한 원-핫 인코딩 수행
# smoker와 activity_level을 범주형으로 처리 (drop_first=True로 다중공선성 방지)
df_encoded = pd.get_dummies(df, columns=['smoker', 'activity_level'], drop_first=True)

# 수치형 변수 스케일링 (로지스틱 회귀 모델의 안정성 확보)
scaler = StandardScaler()
# age와 bmi만 스케일링하고, 나머지는 그대로 둡니다.
df_encoded[['age', 'bmi']] = scaler.fit_transform(df_encoded[['age', 'bmi']])

# --------------------
# 독립변수 식 정의 (Statsmodels formula 형식)
# --------------------
# 모든 인코딩된 특성을 포함하여 formula를 동적으로 생성합니다.
features = ' + '.join(col for col in df_encoded.columns if col not in ['disease'])
formula = f"disease ~ {features}"

print(f"--- 분석에 사용될 Logit Formula: {formula} ---")

--- 분석에 사용될 Logit Formula: disease ~ age + bmi + smoker_1 + activity_level_1 + activity_level_2 ---


In [7]:
# --------------------
# 1) 교차 검증 전 코드 (단일 학습 및 통계 해석)
# --------------------

print("\n\n#####################################################")
print("## 1. 교차 검증 전 (단일 모델 학습 및 해석)")
print("#####################################################")

# 1단계: Statsmodels Logit 모델 학습
model_single_fit = logit(formula, data=df_encoded).fit()

# 2단계: 결과 요약 출력 (OLS와 유사한 표 형태)
print("\n--- [1] Logit Regression Summary Table ---")
print(model_single_fit.summary())

# 3단계: 오즈비 (Odds Ratio) 확인
# 해석의 편의성을 위해 가장 중요한 특성(BMI)과 나머지 특성들을 출력
print("\n--- [2] 회귀 계수와 오즈비 (Odds Ratio) ---")
odds_ratios = np.exp(model_single_fit.params)
odds_ratio_table = pd.DataFrame({
    'Coef. (회귀 계수)': model_single_fit.params,
    'Odds Ratio (오즈비)': odds_ratios
})
# BMI와 흡연 여부(smoker_1)만 출력
# print(odds_ratio_table.loc[['bmi', 'smoker_1', 'activity_level_1', 'activity_level_2']])

# 오즈비 값 해석 예시
print("\n[오즈비 해석: OR > 1 이면 질병 발생 가능성 증가]")
print(f"BMI OR: {np.exp(model_single_fit.params['bmi']):.4f}")



#####################################################
## 1. 교차 검증 전 (단일 모델 학습 및 해석)
#####################################################
Optimization terminated successfully.
         Current function value: 0.636888
         Iterations 5

--- [1] Logit Regression Summary Table ---
                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      994
Method:                           MLE   Df Model:                            5
Date:                Fri, 17 Oct 2025   Pseudo R-squ.:                 0.06005
Time:                        01:09:10   Log-Likelihood:                -636.89
converged:                       True   LL-Null:                       -677.58
Covariance Type:            nonrobust   LLR p-value:                 4.313e-16
                               coef    std err          z      P>|z|      [0.025  

In [8]:
# --------------------
# 2) 교차 검증 후 코드 (K-Fold를 통한 일반화 성능 평가)
# --------------------

print("\n\n#####################################################")
print("## 2. 교차 검증 후 (일반화 성능 평가)")
print("#####################################################")

# 종속변수/독립변수를 Scikit-learn 형식으로 준비
y = df_encoded['disease']
X = df_encoded.drop('disease', axis=1)

# 1단계: Stratified K-Fold 설정 (K=5)
# 분류 문제이므로 클래스 비율을 유지하는 StratifiedKFold 사용
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2단계: Scikit-learn 로지스틱 회귀 모델 정의
model_cv = LogisticRegression(random_state=42)

# 3단계: cross_val_score 함수로 교차 검증 실행
cv_scores = cross_val_score(
    model_cv,       # Scikit-learn 모델
    X, y,           # 전체 데이터
    cv=skf,         # K-Fold 설정
    scoring='accuracy', # 평가 지표
    n_jobs=-1
)

# 4단계: 최종 결과 출력
mean_cv_accuracy = cv_scores.mean()
std_cv_accuracy = cv_scores.std()

print("\n--- K-Fold 교차 검증 (5-Fold) 결과 ---")
print(f"각 Fold의 정확도: {cv_scores}")
print(f"교차 검증 평균 정확도: {mean_cv_accuracy:.4f}")
print(f"정확도 표준 편차: {std_cv_accuracy:.4f}")

# 해석: 이 평균 정확도 값은 모델이 새로운 데이터에 대해 기대할 수 있는 성능을 객관적으로 나타냅니다.



#####################################################
## 2. 교차 검증 후 (일반화 성능 평가)
#####################################################

--- K-Fold 교차 검증 (5-Fold) 결과 ---
각 Fold의 정확도: [0.575 0.65  0.61  0.63  0.6  ]
교차 검증 평균 정확도: 0.6130
정확도 표준 편차: 0.0256


In [ ]:
# ---------------------------------------------------------------
# train, test 

In [9]:
# 라이브러리 및 데이터 불러오기
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
#--------------------------------------------
# EDA
#--------------------------------------------
# train.shape, test.shape
# train.head()
# train.info()
# train.describe()
# import matplotlib.pyplot as plt
# import seaborn as sns
# sns.displot(train['Item_Outlet_Sales'])
# plt.show()
# train.isnull().sum()
# test.isnull().sum()
#--------------------------------------------
# 데이터 전처리
#--------------------------------------------
cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']
target = train.pop('Item_Outlet_Sales')

#
df = pd.concat([train, test])
# 레이블 인코딩
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in cols:
    df[col] = le.fit_transform(df[col])
    
train = df.iloc[:len(train)].copy()
test = df.iloc[len(train):].copy()
train.shape, test.shape

train['Item_Weight'] = train['Item_Weight'].fillna(train['Item_Weight'].min())
train['Outlet_Size'] = train['Outlet_Size'].fillna(train['Outlet_Size'].mode()[0])

test['Item_Weight'] = test['Item_Weight'].fillna(train['Item_Weight'].min())
test['Outlet_Size'] = test['Outlet_Size'].fillna(train['Outlet_Size'].mode()[0])

print(train.shape, test.shape)
train.drop('Item_Identifier', axis=1, inplace=True)
test.drop('Item_Identifier', axis=1, inplace=True)
print(train.shape, test.shape)
#--------------------------------------------
# 검증 데이터 나누기
#--------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=0)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

#--------------------------------------------
# 머신러닝 학습 및 평가
#--------------------------------------------
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error

# LightGBM
import lightgbm as lgb
model = lgb.LGBMRegressor(random_state=0, verbose=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

result = mean_squared_error(y_val, y_pred)
print('MSE:', result)

result = mean_absolute_error(y_val, y_pred)
print('MAE:', result)

result = r2_score(y_val, y_pred)
print('R2:', result)

result = root_mean_squared_error(y_val, y_pred)
print('RMSE:', result)

(6818, 11) (1705, 11)
(6818, 10) (1705, 10)
MSE: 1115654.3482227568
MAE: 736.6367966578568
R2: 0.5702489079618556
RMSE: 1056.2454015155554


In [10]:
#--------------------------------------------
# 머신러닝 학습 및 평가 (cross_val_score를 사용한 K-Fold 교차 검증 - NumPy 대신 statistics 사용)
#--------------------------------------------
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error, make_scorer
# import numpy as np  # NumPy를 사용하지 않음
import statistics # 표준 라이브러리 statistics 모듈 사용
import lightgbm as lgb
import warnings

# LightGBM에서 verbose=-1 설정 시 경고가 발생할 수 있어 무시합니다.
warnings.filterwarnings('ignore', category=UserWarning)


# K-Fold 설정 (예: 5-Fold)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# LightGBM 모델 생성
model = lgb.LGBMRegressor(random_state=0, verbose=-1)

# cross_val_score를 사용하여 각 지표의 평균 점수를 계산합니다.
# scikit-learn은 점수를 최대화하는 방식으로 동작하므로, 
# MSE와 MAE에는 'neg_' 접두사를 붙여 음수로 변환한 후, 결과를 다시 양수로 바꿉니다.

# 1. MSE (Negated Mean Squared Error)
# cross_val_score는 결과를 numpy 배열로 반환하지만, 이후에 리스트로 변환하여 처리합니다.
mse_scores_neg = cross_val_score(
    model, 
    train, 
    target, 
    cv=kf, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
).tolist() # NumPy 배열을 Python 리스트로 변환

# 2. RMSE (MSE를 루트 씌움)
# 각 MSE 점수에 대해 루트를 씌우고, 결과를 리스트로 저장합니다.
rmse_scores = [val**0.5 for val in [-score for score in mse_scores_neg]]

# 3. MAE (Negated Mean Absolute Error)
mae_scores_neg = cross_val_score(
    model, 
    train, 
    target, 
    cv=kf, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
).tolist() # NumPy 배열을 Python 리스트로 변환

# 4. R2 (R-squared)
r2_scores = cross_val_score(
    model, 
    train, 
    target, 
    cv=kf, 
    scoring='r2', 
    n_jobs=-1
).tolist() # NumPy 배열을 Python 리스트로 변환

# 최종 평균 성능 지표 출력
print("="*50)
print(f"Final Average Cross-Validation Results ({n_splits} Folds) using cross_val_score:")
print("="*50)

# MSE (음수 결과를 다시 양수로 변환하고 statistics 사용)
mse_scores = [-score for score in mse_scores_neg]
avg_mse = statistics.mean(mse_scores)
std_mse = statistics.stdev(mse_scores)
print(f'Average MSE: {avg_mse:.4f} (Std: {std_mse:.4f})')

# RMSE
avg_rmse = statistics.mean(rmse_scores)
std_rmse = statistics.stdev(rmse_scores)
print(f'Average RMSE: {avg_rmse:.4f} (Std: {std_rmse:.4f})')

# MAE (음수 결과를 다시 양수로 변환하고 statistics 사용)
mae_scores = [-score for score in mae_scores_neg]
avg_mae = statistics.mean(mae_scores)
std_mae = statistics.stdev(mae_scores)
print(f'Average MAE: {avg_mae:.4f} (Std: {std_mae:.4f})')

# R2
avg_r2 = statistics.mean(r2_scores)
std_r2 = statistics.stdev(r2_scores)
print(f'Average R2: {avg_r2:.4f} (Std: {std_r2:.4f})')
print("="*50)

Final Average Cross-Validation Results (5 Folds) using cross_val_score:
Average MSE: 1223123.3885 (Std: 112737.0744)
Average RMSE: 1105.0079 (Std: 51.0021)
Average MAE: 767.9501 (Std: 29.6811)
Average R2: 0.5790 (Std: 0.0336)
